# Proyecto Profesional de ML: Predicción del Precio de Casas
Este notebook implementa un pipeline modular de Machine Learning para simular datos inmobiliarios, entrenar un modelo de regresión lineal y exportarlo para su uso posterior.

In [1]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir ruta de guardado
folder_path = '/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML'
os.makedirs(folder_path, exist_ok=True)
data_path = os.path.join(folder_path, 'datos.pkl')

# 3. Generación de un dataset simulado
np.random.seed(42)
n_samples = 500

# Características: Metros Cuadrados, Habitaciones, Antigüedad (años)
metros_cuadrados = np.random.normal(120, 40, n_samples).clip(40, 400)
habitaciones = np.random.randint(1, 6, n_samples)
antiguedad = np.random.randint(0, 50, n_samples)

# Variable objetivo: Precio de la casa (con un poco de ruido aleatorio)
# Fórmula base: Precio = 1500 * Metros_Cuadrados + 10000 * Habitaciones - 800 * Antiguedad + 50000 + Ruido
ruido = np.random.normal(0, 15000, n_samples)
precio = (1500 * metros_cuadrados) + (10000 * habitaciones) - (800 * antiguedad) + 50000 + ruido

# Crear DataFrame
df = pd.DataFrame({
    'Metros_Cuadrados': metros_cuadrados,
    'Habitaciones': habitaciones,
    'Antiguedad': antiguedad,
    'Precio': precio
})

# Guardar datos en Google Drive en formato pickle
df.to_pickle(data_path)
print(f"Dataset simulado correctamente y guardado en: {data_path}")
display(df.head())

Mounted at /content/drive
Dataset simulado correctamente y guardado en: /content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML/datos.pkl


,Metros_Cuadrados,Habitaciones,Antiguedad,Precio
0,139.868566,2,9,253182.627045
1,114.469428,2,29,235916.543740
2,145.907542,3,24,272645.794266
3,180.921194,3,38,326179.349610
4,110.633865,5,19,250046.988833


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Cargar datos desde Google Drive
data_path = '/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML/datos.pkl'
df = pd.read_pickle(data_path)

# 2. Separar características (X) y etiqueta objetivo (y)
X = df[['Metros_Cuadrados', 'Habitaciones', 'Antiguedad']]
y = df['Precio']

# 3. División de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Inicializar y entrenar el modelo de Regresión Lineal
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Evaluar el modelo
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("¡Modelo entrenado exitosamente!")
print(f"Error Cuadrático Medio (MSE): {mse:.2f}")
print(f"Coeficiente de Determinación (R2 Score): {r2:.4f}")

¡Modelo entrenado exitosamente!
Error Cuadrático Medio (MSE): 180449308.92
Coeficiente de Determinación (R2 Score): 0.9511


In [3]:
import joblib
import os

# Definir ruta de exportación del modelo
model_path = '/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML/modelo_casas.pkl'

# Exportar modelo
joblib.dump(model, model_path)
print(f"Modelo exportado y guardado exitosamente en: {model_path}")

Modelo exportado y guardado exitosamente en: /content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML/modelo_casas.pkl


In [4]:
import os
import shutil

# 1. Definir rutas
project_folder = '/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML'
zip_output_name = '/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML'
script_path = os.path.join(project_folder, 'usar_modelo.py')

# 2. Escribir el script usar_modelo.py dentro de la misma carpeta de Drive para empaquetarlo todo junto
script_content = """import os
import joblib

# CONFIGURACIÓN: Reemplaza esta ruta por la de tu carpeta de OneDrive en tu PC
# Ejemplo: 'C:/Users/TuUsuario/OneDrive/Proyecto_Casas_ML/modelo_casas.pkl'
RUTA_MODELO = "modelo_casas.pkl" # Por defecto busca en la misma carpeta donde ejecutas el script

def cargar_modelo():
    if not os.path.exists(RUTA_MODELO):
        print(f"Error: No se encontró el archivo del modelo '{RUTA_MODELO}' en el directorio actual.")
        print("Por favor, asegúrate de colocar este script en la misma carpeta de tu OneDrive junto a 'modelo_casas.pkl'.")
        return None
    return joblib.load(RUTA_MODELO)

def main():
    print("=========================================")
    print(" Sistema de Predicción de Precios de Casas")
    print("=========================================")

    model = cargar_modelo()
    if model is None:
        return

    try:
        # Solicitar datos al usuario
        metros = float(input("Introduce los metros cuadrados de la casa: "))
        habitaciones = int(input("Introduce el número de habitaciones: "))
        antiguedad = float(input("Introduce la antigüedad de la casa (en años): "))

        # Preparar los datos para la predicción [[Metros, Habitaciones, Antiguedad]]
        caracteristicas = [[metros, habitaciones, antiguedad]]

        # Realizar predicción
        prediccion = model.predict(caracteristicas)[0]

        print("\\n================ Predicción ================")
        print(f"El precio estimado de la propiedad es: ${prediccion:,.2f} USD")
        print("============================================")

    except ValueError:
        print("\\n[Error]: Por favor introduce solo valores numéricos válidos.")

if __name__ == '__main__':
    main()
"""

with open(script_path, 'w', encoding='utf-8') as f:
    f.write(script_content)

# 3. Comprimir toda la carpeta del proyecto en un archivo .zip
shutil.make_archive(zip_output_name, 'zip', project_folder)

print(f"¡Completado! El archivo ZIP se ha creado exitosamente en tu Google Drive.")
print(f"Ruta del archivo: {zip_output_name}.zip")
print("Ya puedes descargarlo, descomprimirlo en tu OneDrive y ejecutar usar_modelo.py en tu terminal.")

¡Completado! El archivo ZIP se ha creado exitosamente en tu Google Drive.
Ruta del archivo: /content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML.zip
Ya puedes descargarlo, descomprimirlo en tu OneDrive y ejecutar usar_modelo.py en tu terminal.


In [6]:
import os

# 1. Definir rutas
project_folder = '/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML'
output_text_file = os.path.join(project_folder, 'resumen_proyecto_para_ia.txt')

# 2. Plantilla con el contexto de tu proyecto, su estructura y códigos fuente integrados
prompt_contexto = """=====================================================================
SOLICITUD DE GENERACIÓN DE README.MD PROFESIONAL PARA GITHUB
=====================================================================

Hola. Quiero que actúes como un Ingeniero de Software e Ingeniero de Machine Learning experto.
Por favor, genérame un archivo 'README.md' sumamente profesional, con un diseño visualmente atractivo,
iconografía con emojis, explicaciones técnicas y secciones claras para mi portafolio de GitHub.

---------------------------------------------------------------------
INFORMACIÓN GENERAL DEL PROYECTO:
---------------------------------------------------------------------
* Nombre del Proyecto: Sistema de Predicción de Precios de Casas (Machine Learning Clásico)
* Tipo de Modelo: Regresión Lineal de Scikit-Learn (Supervisado)
* Características de entrada (Features):
  1. Metros Cuadrados (Numérico)
  2. Número de Habitaciones (Numérico)
  3. Antigüedad en años (Numérico)
* Variable Objetivo (Target): Precio de la casa (Numérico, simulado mediante una fórmula con ruido aleatorio)
* Rendimiento Obtenido: Coeficiente de Determinación R² de 0.9511 (95.1% de precisión en el set de prueba)
* Enfoque del Proyecto: Modular, escalable y preparado para simulación en la nube (Google Colab Pro) con ejecución local integrada (VS Code mediante OneDrive).

---------------------------------------------------------------------
ESTRUCTURA DEL REPOSITORIO SUGERIDA:
---------------------------------------------------------------------
├── datos.pkl               # Dataset simulado guardado
├── modelo_casas.pkl         # Modelo entrenado exportado con Joblib
├── usar_modelo.py          # Script de Python interactivo para terminal local (VS Code)
├── notebook_entrenamiento.ipynb # Notebook con la preparación y entrenamiento
└── README.md               # Documentación del proyecto (Este archivo)

---------------------------------------------------------------------
CÓDIGO FUENTE UTILIZADO EN CADA MÓDULO:
---------------------------------------------------------------------

[MÓDULO 1: Simulación de Datos y Almacenamiento]
import numpy as np
import pandas as pd
# Simulación de 500 muestras
np.random.seed(42)
n_samples = 500
metros_cuadrados = np.random.normal(120, 40, n_samples).clip(40, 400)
habitaciones = np.random.randint(1, 6, n_samples)
antiguedad = np.random.randint(0, 50, n_samples)
ruido = np.random.normal(0, 15000, n_samples)
precio = (1500 * metros_cuadrados) + (10000 * habitaciones) - (800 * antiguedad) + 50000 + ruido
df = pd.DataFrame({'Metros_Cuadrados': metros_cuadrados, 'Habitaciones': habitaciones, 'Antiguedad': antiguedad, 'Precio': precio})

[MÓDULO 2: Entrenamiento del Modelo de Regresión Lineal]
# Algoritmo: LinearRegression()
# División de datos: 80% Entrenamiento, 20% Test (Test size = 0.2)
# Métricas obtenidas:
# - Error Cuadrático Medio (MSE): 180,449,308.92
# - Coeficiente R²: 0.9511

[MÓDULO 3: Script Interactivo Local (usar_modelo.py)]
# Este script interactúa por consola local pidiéndole datos al usuario
# e importando el modelo binario 'modelo_casas.pkl'

---------------------------------------------------------------------
REQUERIMIENTO DE SECCIONES PARA EL README.MD:
---------------------------------------------------------------------
Por favor, incluye las siguientes secciones en el archivo markdown final:
1. 🌟 Título del Proyecto y breve descripción de impacto.
2. 📊 Características del Dataset (Explicando la procedencia sintética y los campos).
3. ⚙️ Requisitos previos e Instalación (Librerías necesarias como scikit-learn, joblib, pandas, numpy).
4. 🚀 Instrucciones de Ejecución (Cómo usar el notebook en Colab y cómo ejecutar el script local 'usar_modelo.py' mediante la consola).
5. 📈 Rendimiento y Resultados (Resaltando el R² de 95% obtenido de manera clara).
6. 📁 Estructura del repositorio en formato de árbol.
7. 👤 Datos de contacto o autoría.

Haz que el código para copiar el README sea sumamente fácil de copiar.
====================================================================="""

# 3. Guardar el prompt estructurado en un archivo
with open(output_text_file, 'w', encoding='utf-8') as f:
    f.write(prompt_contexto)

print("¡Listo! El archivo para copiar y pegar en tu IA favorita se ha creado en:")
print(output_text_file)
print("\nSolo abre el archivo 'resumen_proyecto_para_ia.txt' en tu Google Drive, copia su contenido y envíaselo a ChatGPT, Claude, Gemini o cualquier otra IA para que te redacte un README de primer nivel.")

¡Listo! El archivo para copiar y pegar en tu IA favorita se ha creado en:
/content/drive/MyDrive/Cursos de IA/Proyecto_Casas_ML/resumen_proyecto_para_ia.txt

Solo abre el archivo 'resumen_proyecto_para_ia.txt' en tu Google Drive, copia su contenido y envíaselo a ChatGPT, Claude, Gemini o cualquier otra IA para que te redacte un README de primer nivel.


## Script para VS Code Local (`usar_modelo.py`)

Copia y guarda el siguiente script en tu computadora para ejecutarlo de forma local. Asegúrate de modificar la ruta para que apunte a tu carpeta sincronizada de OneDrive donde decidas colocar el archivo `modelo_casas.pkl`.

```python
import os
import joblib

# CONFIGURACIÓN: Reemplaza esta ruta por la de tu carpeta de OneDrive en tu PC
# Ejemplo: 'C:/Users/TuUsuario/OneDrive/Proyecto_Casas_ML/modelo_casas.pkl'
RUTA_MODELO = "Escribe_aqui_tu_ruta_de_OneDrive/modelo_casas.pkl"

def cargar_modelo():
    if not os.path.exists(RUTA_MODELO):
        print(f"Error: No se encontró el archivo del modelo en {RUTA_MODELO}")
        print("Por favor, asegúrate de haber copiado el archivo .pkl desde tu Google Drive a tu OneDrive local.")
        return None
    return joblib.load(RUTA_MODELO)

def main():
    print("=========================================")
    print(" Sistema de Predicción de Precios de Casas")
    print("=========================================")
    
    model = cargar_modelo()
    if model is None:
        return

    try:
        # Solicitar datos al usuario
        metros = float(input("Introduce los metros cuadrados de la casa: "))
        habitaciones = int(input("Introduce el número de habitaciones: "))
        antiguedad = float(input("Introduce la antigüedad de la casa (en años): "))
        
        # Preparar los datos para la predicción [[Metros, Habitaciones, Antiguedad]]
        caracteristicas = [[metros, habitaciones, antiguedad]]
        
        # Realizar predicción
        prediccion = model.predict(caracteristicas)[0]
        
        print("\n================ Predicción ================")
        print(f"El precio estimado de la propiedad es: ${prediccion:,.2f} USD")
        print("============================================")
    
    except ValueError:
        print("\n[Error]: Por favor introduce solo valores numéricos válidos.")

if __name__ == "__main__":
    main()
```